<a href="https://colab.research.google.com/github/DymaStar/DTA_2026/blob/main/ML/DS_150626_CW2_ML_feature_engineering_categorical_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature engineering. Categorical features: One-Hot, Ordinal

## Налаштування та дані

- квартири (регресія) — з числовими та категорійними ознаками й датою

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# ---------- Датасет 1: КВАРТИРИ (регресія) ----------
n = 1500
cities = np.random.choice(["Київ", "Львів", "Харків", "Одеса"], n, p=[.4, .2, .2, .2])
city_premium = pd.Series({"Київ": 60, "Львів": 25, "Харків": 10, "Одеса": 20})

condition = np.random.choice(["аварійний", "житловий", "хороший", "євроремонт"],
                             n, p=[.1, .4, .35, .15])
cond_bonus = pd.Series({"аварійний": -20, "житловий": 0, "хороший": 15, "євроремонт": 40})

area    = np.random.normal(60, 20, n).clip(20, 140)
rooms   = np.clip(np.round(area / 25 + np.random.normal(0, .6, n)), 1, 5).astype(int)
floor   = np.random.randint(1, 25, n)
dist_km = np.random.exponential(5, n).clip(.3, 25)
listing_date = pd.to_datetime("2024-01-01") + pd.to_timedelta(
    np.random.randint(0, 540, n), unit="D")

price = (40 + area*1.8 + rooms*5 + floor*.4 - dist_km*3
         + city_premium[cities].values + cond_bonus[condition].values
         + np.random.normal(0, 12, n)).clip(20, None)

apt = pd.DataFrame({
    "area": area.round(1), "rooms": rooms, "floor": floor,
    "dist_km": dist_km.round(1), "city": cities, "condition": condition,
    "listing_date": listing_date, "price": price.round(1),
})


print("Квартири:", apt.shape)
apt.head()

Квартири: (1500, 8)


,area,rooms,floor,dist_km,city,condition,listing_date,price
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1


## Feature Engineering — створення нових ознак

In [3]:
df = apt.copy()
df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1


In [4]:
# Створюємо нову ознаку:
# площа квартири поділена на кількість кімнат

df['area_per_room'] = (df.area / df.rooms).round(1)

# Виводимо перші 5 рядків датафрейму
df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price,area_per_room
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7,27.1
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3,24.1
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1,18.4
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1,32.7
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1,28.1


In [5]:
# Створюємо interaction feature (ознаку взаємодії)

df['area_x_dist'] = (df.area * df.dist_km).round(1)

# Виводимо кілька колонок для перевірки
df[['area', 'dist_km', 'area_x_dist']].head()

,area,dist_km,area_x_dist
0,81.2,4.4,357.3
1,72.3,4.0,289.2
2,73.7,1.2,88.4
3,32.7,2.8,91.6
4,84.2,5.3,446.3


In [6]:
# Взаємодія:
# пошук великої квартири, яка знаходиться далеко від центру

# Створюємо interaction feature
df['area_x_dist'] = (df.area * df.dist_km).round(1)

# Виводимо перші 5 рядків
print("=== Перші 5 рядків ===")
display(df[['area', 'dist_km', 'area_x_dist']].head())

# Статистичний опис
stats = df[['area', 'dist_km', 'area_x_dist']].describe()

print("\n=== Статистичний опис ===")
display(stats)

# Окремий вивід для count
print("\nCOUNT")
print("Кількість значень:")
print(stats.loc['count'])

# Окремий вивід для mean
print("\nMEAN")
print("Середнє значення:")
print(stats.loc['mean'])

# Окремий вивід для std
print("\nSTD")
print("Стандартне відхилення:")
print(stats.loc['std'])

# Окремий вивід для min
print("\nMIN")
print("Мінімальні значення:")
print(stats.loc['min'])

# Окремий вивід для 25%
print("\n25%")
print("25% значень менші за:")
print(stats.loc['25%'])

# Окремий вивід для 50%
print("\n50%")
print("Медіана:")
print(stats.loc['50%'])

# Окремий вивід для 75%
print("\n75%")
print("75% значень менші за:")
print(stats.loc['75%'])

# Окремий вивід для max
print("\nMAX")
print("Максимальні значення:")
print(stats.loc['max'])

=== Перші 5 рядків ===


,area,dist_km,area_x_dist
0,81.2,4.4,357.3
1,72.3,4.0,289.2
2,73.7,1.2,88.4
3,32.7,2.8,91.6
4,84.2,5.3,446.3



=== Статистичний опис ===


,area,dist_km,area_x_dist
count,1500.000000,1500.000000,1500.000000
mean,60.012067,4.689333,281.348067
std,19.851969,4.516440,298.656168
min,20.000000,0.300000,6.000000
25%,45.600000,1.475000,74.150000
50%,59.750000,3.200000,182.050000
75%,73.400000,6.400000,377.150000
max,138.500000,25.000000,1880.000000



COUNT
Кількість значень:
area           1500.0
dist_km        1500.0
area_x_dist    1500.0
Name: count, dtype: float64

MEAN
Середнє значення:
area            60.012067
dist_km          4.689333
area_x_dist    281.348067
Name: mean, dtype: float64

STD
Стандартне відхилення:
area            19.851969
dist_km          4.516440
area_x_dist    298.656168
Name: std, dtype: float64

MIN
Мінімальні значення:
area           20.0
dist_km         0.3
area_x_dist     6.0
Name: min, dtype: float64

25%
25% значень менші за:
area           45.600
dist_km         1.475
area_x_dist    74.150
Name: 25%, dtype: float64

50%
Медіана:
area            59.75
dist_km          3.20
area_x_dist    182.05
Name: 50%, dtype: float64

75%
75% значень менші за:
area            73.40
dist_km          6.40
area_x_dist    377.15
Name: 75%, dtype: float64

MAX
Максимальні значення:
area            138.5
dist_km          25.0
area_x_dist    1880.0
Name: max, dtype: float64


In [10]:
# =========================================================
# Binary Feature
# Квартира близько до центру чи ні
# =========================================================

# Якщо квартира знаходиться ближче ніж 3 км:
# True -> 1
# False -> 0

df['is_central'] = (df.dist_km < 3).astype(int)

# Виводимо перші 5 рядків
print("=== Перші 5 рядків ===")
display(df[['dist_km', 'is_central']].head())

# Кількість значень 0 та 1
print("\n=== Кількість квартир ===")
print(df['is_central'].value_counts())

# Відсотковий розподіл
print("\n=== Відсотковий розподіл ===")
print(round(df['is_central'].value_counts(normalize=True) * 100, 2))

# Висновок:
# Було створено бінарну ознаку is_central.
# Вона показує, чи знаходиться квартира
# близько до центру міста.
# Бінарні ознаки часто використовуються
# у машинному навчанні.

=== Перші 5 рядків ===


,dist_km,is_central
0,4.4,0
1,4.0,0
2,1.2,1
3,2.8,1
4,5.3,0



=== Кількість квартир ===
is_central
0    805
1    695
Name: count, dtype: int64

=== Відсотковий розподіл ===
is_central
0    53.67
1    46.33
Name: proportion, dtype: float64


In [7]:
# Бінарна ознака:
# True(1) - квартира близько до центру
# False(0) - квартира далеко від центру

df['is_central'] = df.dist_km < 3

# Виводимо перші рядки
print("=== Перші 5 рядків ===")
display(df[['dist_km', 'is_central']].head())

# Окремо показуємо True та False
print("\n=== Кількість True / False ===")
print(df['is_central'].value_counts())

# Кількість квартир близько до центру
print("\n=== Квартири близько до центру ===")
print((df['is_central'] == True).sum())

# Кількість квартир далеко від центру
print("\n=== Квартири далеко від центру ===")
print((df['is_central'] == False).sum())

# Відсоток квартир близько до центру
print("\n=== Відсоток квартир близько до центру ===")
print(round(df['is_central'].mean() * 100, 2), "%")

=== Перші 5 рядків ===


,dist_km,is_central
0,4.4,False
1,4.0,False
2,1.2,True
3,2.8,True
4,5.3,False



=== Кількість True / False ===
is_central
False    805
True     695
Name: count, dtype: int64

=== Квартири близько до центру ===
695

=== Квартири далеко від центру ===
805

=== Відсоток квартир близько до центру ===
46.33 %


In [11]:
# =========================================================
# Binning / Categorization
# Поділ поверхів на категорії
# =========================================================

# Ділимо поверхи на групи:
# low    -> низькі поверхи
# middle -> середні поверхи
# high   -> високі поверхи

df['floor_group'] = pd.cut(
    df.floor,
    bins=[0, 2, 9, 100],
    labels=['low', 'middle', 'high']
)

# Виводимо перші 5 рядків
print("=== Перші 5 рядків ===")
display(df[['floor', 'floor_group']].head())

# Кількість квартир у кожній категорії
print("\n=== Кількість квартир ===")
print(df['floor_group'].value_counts())

# Відсотковий розподіл
print("\n=== Відсотковий розподіл ===")
print(round(df['floor_group'].value_counts(normalize=True) * 100, 2))

# Висновок:
# Було виконано binning ознаки floor.
# Поверхи були поділені на категорії:
# low, middle та high.
# Це дозволяє спростити аналіз даних
# та підготувати категоріальні ознаки для ML.

=== Перші 5 рядків ===


,floor,floor_group
0,20,high
1,10,high
2,23,high
3,9,middle
4,2,low



=== Кількість квартир ===
floor_group
high      960
middle    404
low       136
Name: count, dtype: int64

=== Відсотковий розподіл ===
floor_group
high      64.00
middle    26.93
low        9.07
Name: proportion, dtype: float64


In [12]:
# =========================
# Створення ознак з дати
# =========================

# list_month — номер місяця
# Наприклад:
# January = 1
# February = 2
# ...
# December = 12
df['list_month'] = df.listing_date.dt.month

# list_weekday — день тижня
# Monday = 0
# Tuesday = 1
# ...
# Sunday = 6
df['list_weekday'] = df.listing_date.dt.weekday

# is_weekend — чи є день вихідним
# True/False перетворюємо в 1/0
# Saturday = 5
# Sunday = 6
df['is_weekend'] = (df['list_weekday'] >= 5).astype(int)

# Перевіряємо результат
df[['listing_date', 'list_month', 'list_weekday', 'is_weekend']].head()

,listing_date,list_month,list_weekday,is_weekend
0,2024-04-23,4,1,0
1,2025-01-31,1,4,0
2,2024-06-22,6,5,1
3,2024-09-11,9,2,0
4,2025-04-10,4,3,0


In [13]:
# Виводимо вибрані ознаки після feature engineering
df[[
    'area',
    'rooms',
    'area_per_room',
    'dist_km',
    'is_central',
    'floor',
    'floor_group',
    'list_month',
    'is_weekend'
]].head()


,area,rooms,area_per_room,dist_km,is_central,floor,floor_group,list_month,is_weekend
0,81.2,3,27.1,4.4,0,20,high,4,0
1,72.3,3,24.1,4.0,0,10,high,1,0
2,73.7,4,18.4,1.2,1,23,high,6,1
3,32.7,1,32.7,2.8,1,9,middle,9,0
4,84.2,3,28.1,5.3,0,2,low,4,0


1. corr()['price']
   — рахує кореляцію всіх ознак з price.

2. drop('price')
   — прибирає кореляцію price з самим собою (=1).

3. round(3)
   — округлює до 3 знаків.

4. sort_values(key=abs, ascending=False)
   — сортує ознаки за силою зв'язку,
     незалежно від знаку (+ або -).

In [17]:
# Кореляція ознак з price
check = df[[
    'area',
    'rooms',
    'area_per_room',
    'area_x_dist',
    'dist_km',
    'is_central',
    'floor',
    'price'
]].corr()['price']

check.drop('price').round(3).sort_values(key=abs, ascending=False)

,price
area,0.779
rooms,0.645
dist_km,-0.255
is_central,0.181
area_per_room,-0.052
area_x_dist,0.013
floor,0.010


In [18]:
# ==========================================
# One-Hot Encoding для категоріальних ознак
# ==========================================

# city і condition — це категоріальні ознаки (текстові значення)
# Модель ML не вміє працювати з текстом напряму,
# тому перетворюємо категорії у числові прапорці (0/1)

dummies = pd.get_dummies(
    df[['city', 'condition']],

    # dtype=int:
    # False/True будуть перетворені у 0/1
    dtype=int
)

# Виводимо інформацію:
# скільки було колонок і скільки стало після One-Hot
print(f'Було 2 стовпчики, стало {dummies.shape[1]} числових прапорців')

# Показуємо перші рядки
dummies.head()

Було 2 стовпчики, стало 8 числових прапорців


,city_Київ,city_Львів,city_Одеса,city_Харків,condition_аварійний,condition_житловий,condition_хороший,condition_євроремонт
0,1,0,0,0,0,0,1,0
1,0,0,1,0,0,1,0,0
2,0,0,0,1,1,0,0,0
3,0,1,0,0,0,1,0,0
4,1,0,0,0,0,1,0,0


In [20]:
# ==========================================
# OneHotEncoder зі sklearn
# ==========================================

# Імпортуємо OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

# Створюємо encoder
# handle_unknown='ignore':
# якщо в тестових даних з'явиться нове місто,
# модель не видасть помилку

# sparse_output=False:
# результат буде звичайним DataFrame/масивом,
# а не sparse matrix
ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

# Навчаємо encoder на колонці city
# і одночасно перетворюємо категорії у числа
encoded = ohe.fit_transform(df[['city']])

# Виводимо список категорій,
# які знайшов encoder
print(list(ohe.categories_[0]))
# Виводимо перші рядки encoded даних
print('\nПерші рядки:')
# Створюємо DataFrame з encoded даних
# get_feature_names_out() створює назви колонок:
# city_Київ, city_Львів і т.д.
encoded_df = pd.DataFrame(
    encoded[:5],
    columns=ohe.get_feature_names_out(['city'])
).astype(int)

# Виводимо результат
encoded_df

['Київ', 'Львів', 'Одеса', 'Харків']

Перші рядки:


,city_Київ,city_Львів,city_Одеса,city_Харків
0,1,0,0,0
1,0,0,1,0
2,0,0,0,1
3,0,1,0,0
4,1,0,0,0


## Ordinal (порядковий)

In [21]:
# Дивимось унікальні значення condition
df['condition'].unique()

array(['хороший', 'житловий', 'аварійний', 'євроремонт'], dtype=object)

In [22]:
# ==========================================
# Ordinal Encoding для порядкової ознаки
# ==========================================

from sklearn.preprocessing import OrdinalEncoder

# Задаємо порядок категорій:
# від найгіршого стану до найкращого
order = [[
    'аварійний',
    'житловий',
    'хороший',
    'євроремонт'
]]

# Створюємо encoder
oe = OrdinalEncoder(categories=order)

# Копіюємо колонку condition
apt_demo = apt[['condition']].copy()

# Перетворюємо категорії у числа
# 0 -> аварійний
# 1 -> житловий
# 2 -> хороший
# 3 -> євроремонт
apt_demo['condition_code'] = oe.fit_transform(
    apt[['condition']]
).astype(int)

# Видаляємо дублікати
# і сортуємо по condition_code
apt_demo.drop_duplicates().sort_values('condition_code')

,condition,condition_code
2,аварійний,0
1,житловий,1
0,хороший,2
12,євроремонт,3


In [23]:
# Перетворюємо текстову ознаку condition у числову
# та додаємо нову колонку condition_code у DataFrame df

df['condition_code'] = oe.fit_transform(
    apt[['condition']]
).astype(int)

# Перевіряємо результат
df.head()

,area,rooms,floor,dist_km,city,condition,listing_date,price,area_per_room,area_x_dist,is_central,floor_group,list_month,list_weekday,is_weekend,condition_code
0,81.2,3,20,4.4,Київ,хороший,2024-04-23,260.7,27.1,357.3,0,high,4,1,0,2
1,72.3,3,10,4.0,Одеса,житловий,2025-01-31,216.3,24.1,289.2,0,high,1,4,0,1
2,73.7,4,23,1.2,Харків,аварійний,2024-06-22,179.1,18.4,88.4,1,high,6,5,1,0
3,32.7,1,9,2.8,Львів,житловий,2024-09-11,98.1,32.7,91.6,1,middle,9,2,0,1
4,84.2,3,2,5.3,Київ,житловий,2025-04-10,257.1,28.1,446.3,0,low,4,3,0,1
